# Proyecto - Modelo dinámico de un robot 3R

**Proyecto:** Obtener los datos y generar el modelo dinámico de un robot 3R.  
**Software utilizado:** Autodesk Inventor, MATLAB, ROS 2 y RViz.  

---

## 1. Introducción

Durante el desarrollo de este proyecto se trabajó con un manipulador robótico de tres grados de libertad, conocido como robot 3R, en el cual el movimiento se produce mediante tres articulaciones rotacionales. El objetivo principal no fue solamente visualizar el robot, sino tambbién obtener sus parámetros físicos reales, construir su modelo cinemático y generar el modelo dinámico que permita conocer el comportamiento del sistema ante una trayectoria definida.

En robótica, el modelo dinámico es una parte fundamental porque permite relacionar el movimiento de las articulaciones con los torques que deben entregar los actuadores. Esto significa que no basta solamente con conocer la posición del efector final; también es necesario conocer masas, centros de masa, inercias, velocidades, aceleraciones y efectos gravitacionales. Sin este análisis, sería posible proponer movimientos que visualmente parecen correctos, pero que físicamente podrían exigir torques excesivos o generar esfuerzos difíciles de alcanzar por los motores.

Para este proyecto se comenzó a trabajar a partir de un modelo CAD del robot en Autodesk Inventor. A partir del ensamble se obtuvieron las propiedades físicas de cada eslabón: masa, centro de gravedad y matriz de inercia. Posteriormente, estos datos se convirtieron al Sistema Internacional de unidades y se sustituyeron en el modelo matemático desarrollado en MATLAB.

Además, se generó una trayectoria articular suave mediante un polinomio de quinto orden, esto con el propósito de evitar cambios bruscos de velocidad y aceleración. Finalmente, el robot fue visualizado en RViz mediante ROS 2, lo cual comprobó que el modelo URDF y las articulaciones correspondían correctamente con el mecanismo diseñado.

## 2. Desarrollo

Para cumplir con el objetivo del proyecto se dividió el trabajo en varias etapas:

1. Obtención de datos físicos desde Autodesk Inventor.
2. Definición geométrica del robot 3R.
3. Planteamiento de la cinemática directa.
4. Cálculo de centros de masa y velocidades.
5. Obtención del modelo dinámico mediante el método de Lagrange.
6. Evaluación de una trayectoria articular.
7. Visualización del robot en RViz mediante ROS 2.

### 2.1 Datos físicos obtenidos desde Autodesk Inventor

Las propiedades físicas se obtuvieron desde la herramienta de propiedades físicas de Autodesk Inventor. Para evitar errores de referencia, los datos fueron tomados del ensamble y luego se identificaron los eslabones dinámicos del robot.

La base se consideró fija, por lo que no entra directamente en el modelo dinámico. Sin embargo, su masa se utilizó para comprobar la consistencia comparando la masa total calculada con la masa total del ensamble reportada por Inventor.

Los eslabones considerados en el modelo dinámico fueron:

- **Eslabón 1:** cintura.
- **Eslabón 2:** brazo vertical.
- **Eslabón 3:** brazo horizontal con pinza fija.

<div align="center">
  <img src="./Imagenes%20proyecto/propiedades_inventor.png" width="650">
  <br>
  <sub><i>Figura 1: Obtención de propiedades físicas desde Autodesk Inventor (masa total del ensamble).</i></sub>
</div>
<br>

In [1]:
# Datos físicos del robot 3R
# Todas las unidades están en Sistema Internacional.

import numpy as np

# Masas [kg]
m1 = 0.214   # cintura
m2 = 0.448   # brazo vertical
m3 = 0.304   # brazo horizontal + pinza

# Masa de la base, usada solo para validación
m_base = 0.402

# Masa total reportada por Inventor
m_total_inventor = 1.367

# Masa móvil del robot
m_movil = m1 + m2 + m3

# Masa total calculada
m_total_calculada = m_base + m_movil

print("Masa móvil del robot [kg]:", m_movil)
print("Masa total calculada [kg]:", m_total_calculada)
print("Masa total reportada por Inventor [kg]:", m_total_inventor)
print("Diferencia [kg]:", m_total_calculada - m_total_inventor)

Masa móvil del robot [kg]: 0.966
Masa total calculada [kg]: 1.3679999999999999
Masa total reportada por Inventor [kg]: 1.367
Diferencia [kg]: 0.0009999999999998899


La masa móvil total del robot fue:

\[
m_{móvil} = 0.214 + 0.448 + 0.304 = 0.966\ kg
\]

La masa total calculada del ensamble fue:

\[
m_{total} = 0.402 + 0.966 = 1.368\ kg
\]

Mientras que Inventor reportó aproximadamente:

\[
m_{Inventor} = 1.367\ kg
\]

La diferencia entre ambas masas fue mínima, por lo que se validó que las propiedades individuales fueron tomadas correctamente.

### 2.2 Centros de masa e inercias

Los centros de masa absolutos fueron obtenidos de Inventor. Después, se calcularon los centros de masa relativos a cada articulación correspondiente, ya que estos son los que se utilizan para el análisis dinámico.

Los datos se manejaron en metros y kilogramos para mantener consistencia con el Sistema Internacional.

In [ ]:
# Centros de masa absolutos obtenidos de Inventor [m]

rC1_abs = np.array([0.007316, -0.012220, 0.067155])
rC2_abs = np.array([0.007415, -0.010549, 0.205414])
rC3_abs = np.array([0.007799849, -0.161106668, 0.326759046])

# Posiciones aproximadas de las articulaciones en Inventor [m]

rJ1_abs = np.array([0.007316, -0.012207, 0.045191])
rJ2_abs = np.array([0.007316, -0.012207, 0.085000])
rJ3_abs = np.array([0.007316, -0.039207, 0.322000])
rP_abs  = np.array([0.007316, -0.255207, 0.322000])

# Centros de masa relativos a cada articulación [m]

rC1 = rC1_abs - rJ1_abs
rC2 = rC2_abs - rJ2_abs
rC3 = rC3_abs - rJ3_abs

print("rC1 relativo [m]:", rC1)
print("rC2 relativo [m]:", rC2)
print("rC3 relativo [m]:", rC3)

In [ ]:
# Matrices de inercia respecto al centro de gravedad [kg*m^2]

IC1 = np.array([
    [ 0.000068001, -0.000000484, -0.000000060],
    [-0.000000484,  0.000165953, -0.000000033],
    [-0.000000060, -0.000000033,  0.000137221]
])

IC2 = np.array([
    [0.002115146, 0.000000011, 0.000000053],
    [0.000000011, 0.002204573, 0.000008439],
    [0.000000053, 0.000008439, 0.000142491]
])

IC3 = np.array([
    [0.002391018, 0.000009703, 0.000000681],
    [0.000009703, 0.000062846, 0.000068461],
    [0.000000681, 0.000068461, 0.002397553]
])

print("IC1 =\n", IC1)
print("IC2 =\n", IC2)
print("IC3 =\n", IC3)

### 2.3 Geometría del robot

A partir del modelo CAD y la configuración del URDF se identificaron las posiciones de las articulaciones. El robot tiene tres grados de libertad rotacionales:

- \(q_1\): rotación alrededor del eje \(Z\).
- \(q_2\): rotación alrededor del eje \(X\).
- \(q_3\): rotación alrededor del eje \(X\).

Las distancias principales entre articulaciones fueron:

\[
z_{12} = 0.039809\ m
\]

\[
y_{23} = -0.027\ m
\]

\[
z_{23} = 0.237\ m
\]

\[
y_{3P} = -0.216\ m
\]

In [ ]:
# Distancias entre articulaciones [m]

z12 = rJ2_abs[2] - rJ1_abs[2]

y23 = rJ3_abs[1] - rJ2_abs[1]
z23 = rJ3_abs[2] - rJ2_abs[2]

y3P = rP_abs[1] - rJ3_abs[1]
z3P = rP_abs[2] - rJ3_abs[2]

print("z12 =", z12)
print("y23 =", y23)
print("z23 =", z23)
print("y3P =", y3P)
print("z3P =", z3P)

### 2.4 Cinemática directa

La cinemática directa se planteó usando transformaciones homogéneas. El marco base se colocó en la primera articulación del robot.

Las transformaciones utilizadas fueron:

\[
T^0_1 = R_z(q_1)
\]

\[
T^1_2 = T_z(z_{12})R_x(q_2)
\]

\[
T^2_3 = T_y(y_{23})T_z(z_{23})R_x(q_3)
\]

\[
T^3_P = T_y(y_{3P})
\]

Con estas matrices se obtuvo la posición del efector final como función de las variables articulares.

In [ ]:
import sympy as sp

q1, q2, q3 = sp.symbols('q1 q2 q3', real=True)

def RotZ(th):
    return sp.Matrix([
        [sp.cos(th), -sp.sin(th), 0, 0],
        [sp.sin(th),  sp.cos(th), 0, 0],
        [0,           0,          1, 0],
        [0,           0,          0, 1]
    ])

def RotX(th):
    return sp.Matrix([
        [1, 0,           0,          0],
        [0, sp.cos(th), -sp.sin(th), 0],
        [0, sp.sin(th),  sp.cos(th), 0],
        [0, 0,           0,          1]
    ])

def Trans(x, y, z):
    return sp.Matrix([
        [1, 0, 0, x],
        [0, 1, 0, y],
        [0, 0, 1, z],
        [0, 0, 0, 1]
    ])

T0_1 = RotZ(q1)
T1_2 = Trans(0, 0, z12) * RotX(q2)
T2_3 = Trans(0, y23, z23) * RotX(q3)
T3_P = Trans(0, y3P, z3P)

T0_2 = sp.simplify(T0_1 * T1_2)
T0_3 = sp.simplify(T0_2 * T2_3)
T0_P = sp.simplify(T0_3 * T3_P)

pP = sp.simplify(T0_P[:3, 3])

print("Posición del efector final respecto a J1:")
sp.pretty_print(pP)

pP_cero = np.array([float(v) for v in pP.subs({q1:0, q2:0, q3:0})])
pP_abs_cero = rJ1_abs + pP_cero

print("\nPosición absoluta del efector final con q = 0 [m]:")
print(pP_abs_cero)

### 2.5 Modelo dinámico

El modelo dinámico se obtuvo mediante el método de Lagrange. Para ello se calcularon las energías cinética y potencial de cada eslabón.

La energía cinética total se compone de una parte traslacional y una parte rotacional:

\[
K = \sum_{i=1}^{3} \left( \frac{1}{2}m_i v_{Ci}^{T}v_{Ci} + \frac{1}{2}\omega_i^{T}I_i\omega_i \right)
\]

La energía potencial gravitacional se calculó como:

\[
U = \sum_{i=1}^{3} m_i g z_{Ci}
\]

El Lagrangiano se define como:

\[
L = K - U
\]

Aplicando la ecuación de Euler-Lagrange:

\[
\tau_i = \frac{d}{dt}\left(\frac{\partial L}{\partial \dot{q}_i}\right) - \frac{\partial L}{\partial q_i}
\]

se obtiene el vector de torques generalizados.

El resultado final se organizó en la forma:

\[
\tau = M(q)\ddot{q} + V(q,\dot{q}) + G(q)
\]

donde:

- \(M(q)\): matriz de inercia.
- \(V(q,\dot{q})\): términos de velocidad, Coriolis y centrífugos.
- \(G(q)\): vector de gravedad.

Durante la verificación en MATLAB se comprobó que:

\[
\tau - \left(M(q)\ddot{q} + V(q,\dot{q}) + G(q)\right) \approx 0
\]

El error obtenido fue del orden de \(10^{-15}\), por lo que se considera despreciable.

<div align="center">
  <img src="Imagenes%20proyecto/verificacion_dinamica.png" width="150">
  <br>
  <sub><i>Figura 2: Verificación numérica del modelo dinámico en MATLAB.</i></sub>
</div>
<br>

### 2.6 Trayectoria articular

Para validar el modelo dinámico se utilizó una trayectoria articular suave de reposo a reposo mediante un polinomio de quinto orden. Esta trayectoria evita saltos bruscos en velocidad y aceleración, lo cual es importante para no exigir torques irreales a los actuadores.

La función de interpolación fue:

\[
s(t) = 10r^3 - 15r^4 + 6r^5
\]

con:

\[
r = \frac{t}{t_f}
\]

La trayectoria propuesta fue:

\[
q_0 = [0^\circ,\ 0^\circ,\ 0^\circ]^T
\]

\[
q_f = [45^\circ,\ 25^\circ,\ -30^\circ]^T
\]

In [ ]:
import matplotlib.pyplot as plt

tf = 5.0
N = 200
t = np.linspace(0, tf, N)

q0_deg = np.array([0, 0, 0], dtype=float)
qf_deg = np.array([45, 25, -30], dtype=float)

q0 = np.deg2rad(q0_deg)
qf = np.deg2rad(qf_deg)

Q = np.zeros((3, N))
DQ = np.zeros((3, N))
DDQ = np.zeros((3, N))

for i in range(N):
    r = t[i] / tf
    
    s = 10*r**3 - 15*r**4 + 6*r**5
    ds = (30*r**2 - 60*r**3 + 30*r**4) / tf
    dds = (60*r - 180*r**2 + 120*r**3) / tf**2
    
    Q[:, i] = q0 + (qf - q0) * s
    DQ[:, i] = (qf - q0) * ds
    DDQ[:, i] = (qf - q0) * dds

print("Trayectoria generada correctamente.")

## 3. Resultados

A partir del modelo dinámico y la trayectoria articular se obtuvieron las gráficas de posición, velocidad, aceleración, torque, potencia y trayectoria del efector final. Estas gráficas permiten analizar el comportamiento del robot durante el movimiento propuesto.

### 3.1 Posición articular

La posición articular muestra el desplazamiento suave de las tres articulaciones desde la configuración inicial hasta la configuración final. El perfil evita cambios bruscos al inicio y al final gracias al polinomio de quinto orden.

<div align="center">
  <img src="Imagenes%20proyecto/posicion_articular.png" width="450">
  <br>
  <sub><i>Figura 3: Posición articular del robot 3R.</i></sub>
</div>
<br>

### 3.2 Velocidad y aceleración articular

Los perfiles de velocidad presentan forma de campana, lo que indica que el movimiento inicia y termina en reposo. La aceleración también se mantiene continua, lo cual evita cambios instantáneos que podrían exigir esfuerzos mecánicos excesivos.

<div align="center">
  <img src="Imagenes%20proyecto/velocidad_articular.png" width="450">
  <br>
  <sub><i>Figura 4: Velocidad articular del robot 3R.</i></sub>
</div>
<br>

<div align="center">
  <img src="Imagenes%20proyecto/aceleracion_articular.png" width="450">
  <br>
  <sub><i>Figura 5: Aceleración articular del robot 3R.</i></sub>
</div>
<br>

### 3.3 Torque y potencia

Al sustituir la trayectoria en el modelo dinámico se obtuvieron los torques requeridos para cada articulación. Estos valores permiten conocer el esfuerzo que tendrían que entregar los actuadores para ejecutar la trayectoria.

La potencia se calculó mediante:

\[
P_i = \tau_i \dot{q}_i
\]

donde \(P_i\) es la potencia de la articulación \(i\).

<div align="center">
  <img src="Imagenes%20proyecto/torque_articular.png" width="450">
  <br>
  <sub><i>Figura 6: Torque requerido por articulación.</i></sub>
</div>
<br>

<div align="center">
  <img src="Imagenes%20proyecto/potencia_articular.png" width="450">
  <br>
  <sub><i>Figura 7: Potencia mecánica por articulación.</i></sub>
</div>
<br>

### 3.4 Trayectoria del efector final

La trayectoria del efector final se calculó usando la cinemática directa. Aunque la trayectoria se definió en espacio articular, el movimiento resultante permite observar el desplazamiento espacial de la pinza.

<div align="center">
  <img src="Imagenes%20proyecto/trayectoria_efector.png" width="450">
  <br>
  <sub><i>Figura 8: Trayectoria espacial del efector final.</i></sub>
</div>
<br>

### 3.5 Visualización en RViz

El robot fue visualizado en RViz usando ROS 2. Para ello se generó un archivo URDF con los eslabones, articulaciones y mallas STL exportadas desde Autodesk Inventor. Posteriormente, se creó un nodo que publica la trayectoria articular en el tópico `/joint_states`, permitiendo observar el movimiento del robot.

El comando que se utilizó para ejecutar la visualización fue:

```bash
cd ~/clases_robotica/semestre_2026_2
source install/setup.bash
ros2 launch robot_3r_description trayectoria_rviz.launch.xml
```

<div align="center">
  <img src="Imagenes%20proyecto/rviz_robot_3r_1.png" width="400">
  <br>
  <sub><i>Figura 9: Primera postura del robot 3R visualizada en RViz.</i></sub>
</div>
<br>

<div align="center">
  <img src="Imagenes%20proyecto/rviz_robot_3r_2.png" width="400">
  <br>
  <sub><i>Figura 10: Segunda postura del robot 3R visualizada en RViz.</i></sub>
</div>
<br>

El video de la trayectoria articular ejecutada en RViz se encuentra en el enlace:

[Ver video de la trayectoria del robot 3R en RViz](https://youtu.be/Rm0kOz8nyYw?si=5r2N5cO_UWhuIjvp)

## 4. Conclusión

El desarrollo de este proyecto permitió obtener el modelo dinámico de un robot manipulador 3R a partir de datos reales extraídos de Autodesk Inventor. Se identificaron las masas, centros de masa e inercias de los eslabones móviles, y se convirtieron al Sistema Internacional para ser utilizados en el análisis matemático.

A partir de la geometría del robot se planteó la cinemática directa mediante transformaciones homogéneas, considerando que la primera articulación gira alrededor del eje \(Z\), mientras que la segunda y tercera articulación giran alrededor del eje \(X\). Finalmente, se calcularon los centros de masa, velocidades lineales, velocidades angulares, energía cinética, energía potencial y el Lagrangiano del sistema.

El modelo dinámico obtenido se expresó en la forma:


tau = M(q)\ddot{q} + V(q,\dot{q}) + G(q)


La verificación numérica mostró que la reconstrucción del torque coincide con el torque obtenido por el método de Lagrange, con un error del orden de (10^{-15}), por lo que el modelo se considera correctamente formulado.

Finalmente, se generó una trayectoria articular suave mediante un polinomio de quinto orden y se visualizó el movimiento del robot en RViz. Con esto se comprobó que el modelo URDF, la configuración de articulaciones y la trayectoria propuesta funcionan correctamente.

## 5. Fuentes

[1] R. Kelly y V. Santibáñez, *Control de Movimiento de Robots Manipuladores*, 1.ª ed. Madrid, España: Pearson Educación, 2003.

[2] J. J. Craig, *Robótica: Mecánica y Control*, 3.ª ed. Naucalpan de Juárez, México: Pearson Educación, 2006.

[3] M. W. Spong, S. Hutchinson, y M. Vidyasagar, *Robot Modeling and Control*, 1.ª ed. Hoboken, NJ, EE. UU.: John Wiley & Sons, 2005.

[4] B. Siciliano, L. Sciavicco, L. Villani, y G. Oriolo, *Robotics: Modelling, Planning and Control*. Londres, Reino Unido: Springer-Verlag, 2009.